In [8]:
import pandas as pd
import geopandas as gp
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

In [9]:
# location of health export
filepath = '/Users/ryansponzilli/Developer/Python Projects/health-data/data/raw/apple_health/apple_health_export/export.xml'
# create element tree object
tree = ET.parse(filepath) 
# for every health record, extract the attributes
root = tree.getroot()
record_list = [x.attrib for x in root.iter('Record')]

record_data = pd.DataFrame(record_list)

In [10]:
record_data.head()

,type,sourceName,sourceVersion,unit,creationDate,startDate,endDate,value,device
0,HKQuantityTypeIdentifierDietaryWater,WaterMinder,552,mL,2021-01-25 11:00:05 -0600,2021-01-25 11:00:04 -0600,2021-01-25 11:00:04 -0600,0,NaN
1,HKQuantityTypeIdentifierDietaryWater,WaterMinder,544,mL,2020-10-07 14:38:40 -0600,2020-10-07 14:38:40 -0600,2020-10-07 14:38:40 -0600,946.353,NaN
2,HKQuantityTypeIdentifierDietaryWater,WaterMinder,505,mL,2020-08-14 10:07:22 -0600,2020-08-14 10:07:18 -0600,2020-08-14 10:07:18 -0600,354.882,NaN
3,HKQuantityTypeIdentifierDietaryWater,WaterMinder,505,mL,2020-08-14 11:09:33 -0600,2020-08-14 11:09:28 -0600,2020-08-14 11:09:28 -0600,354.882,NaN
4,HKQuantityTypeIdentifierDietaryWater,WaterMinder,505,mL,2020-08-14 11:58:30 -0600,2020-08-14 11:58:30 -0600,2020-08-14 11:58:30 -0600,354.882,NaN


In [11]:
record_data_parsed = record_data.copy()

# proper type to dates
for col in ['creationDate', 'startDate', 'endDate']:
    record_data_parsed[col] = pd.to_datetime(record_data_parsed[col])

# value is numeric, NaN if fails
record_data_parsed['value_num'] = pd.to_numeric(record_data_parsed['value'], errors='coerce')

# some records do not measure anything, just count occurences
# filling with 1.0 (= one time) makes it easier to aggregate
record_data_parsed['value'] = record_data_parsed['value'].fillna(1.0)

# shorter observation names
record_data_parsed['type'] = record_data_parsed['type'].str.replace('HKQuantityTypeIdentifier', '')
record_data_parsed['type'] = record_data_parsed['type'].str.replace('HKCategoryTypeIdentifier', '')

In [12]:
record_data['type'].value_counts()

type
HKQuantityTypeIdentifierActiveEnergyBurned                1694733
HKQuantityTypeIdentifierHeartRate                          684669
HKQuantityTypeIdentifierBasalEnergyBurned                  659559
HKQuantityTypeIdentifierDistanceWalkingRunning             529268
HKQuantityTypeIdentifierStepCount                          459128
HKQuantityTypeIdentifierPhysicalEffort                     221295
HKQuantityTypeIdentifierAppleExerciseTime                   99821
HKQuantityTypeIdentifierAppleStandTime                      89697
HKQuantityTypeIdentifierEnvironmentalAudioExposure          67157
HKQuantityTypeIdentifierWalkingSpeed                        52427
HKQuantityTypeIdentifierWalkingStepLength                   52421
HKCategoryTypeIdentifierAppleStandHour                      50636
HKQuantityTypeIdentifierRespiratoryRate                     47014
HKQuantityTypeIdentifierWalkingDoubleSupportPercentage      45652
HKCategoryTypeIdentifierSleepAnalysis                       43456
HKQua

In [13]:
record_data.loc[record_data['type'] == "RunningSpeed"]

,type,sourceName,sourceVersion,unit,creationDate,startDate,endDate,value,device
